# 12 · Keep data lazy and close its storage cleanly
Read CSV telemetry once, persist the declared AO to Zarr, and plan an analysis without
loading its numerical payload. Then compute deliberately and close the resources.
[Setup](README.md). All output stays in a temporary directory.

In [1]:
from pathlib import Path
from tempfile import TemporaryDirectory

import numpy as np
import pandas as pd
import xarray as xr
from dask.callbacks import Callback
from IPython.display import display

from tal import AnalysisObject
from tal.io import AOZarrReadOptions, CsvIngestOptions, read_csv_logs, write_csv_logs
from tal.spatial import Position

data_dir = Path("data/flight_telemetry")
if not data_dir.is_dir():
    data_dir = Path("examples/tutorial/data/flight_telemetry")
observations = read_csv_logs(str(data_dir / "observations/*.csv"), opts=CsvIngestOptions(time_col="time"))

## A canonical round trip and an explicit computation boundary
Zarr preserves decoded values and TAL schema. Reopening validates small validity
metadata eagerly; data variables remain lazy. The callback below starts **after**
opening so it measures planning, not metadata reads. `compute()` is the deliberate
payload boundary. Close the derived wrapper and source before removing storage.

The installed Zarr 3 writer emits a notice about consolidated-metadata portability. It does not invalidate the decoded round-trip check below.

Assemble a `Position` directly from the reopened fields, take its norm, and
average over each trial's recorded samples. This measures distance from the world
origin, not distance traveled or distance to the deck. Different recording lengths
also mean these averages are not a fair controller ranking. Field assembly and
these reductions build a lazy graph; `compute()` is the deliberate execution boundary.

In [2]:
with TemporaryDirectory(prefix="tal-tutorial-") as temporary:
    output_dir = Path(temporary)
    observations.io.to_zarr(str(output_dir / "flight.zarr"))
    lazy = AnalysisObject.from_zarr(str(output_dir / "flight.zarr"), opts=AOZarrReadOptions(chunks={}))
    try:
        assert lazy.as_dataset(copy="shallow")["altitude"].chunks is not None
        planning_tasks = []
        with Callback(posttask=lambda key, *args: planning_tasks.append(key)):
            lazy_position = Position.from_fields(lazy, "drone.position.{x,y,z}", parent="world", child="drone")
            planned = lazy_position.norm().mean(dim="sample")
        assert planning_tasks == []
        display(planned)
        try:
            result = planned.as_dataset(copy="shallow").compute()
            decoded = lazy.as_dataset(copy="shallow").compute()
        finally:
            planned.close()
            lazy_position.close()
        xr.testing.assert_identical(decoded, observations.as_dataset(copy="shallow"))
    finally:
        lazy.close()
    csv_paths = write_csv_logs(observations.select_vars("altitude"), str(output_dir / "csv"))
    exported = pd.read_csv(csv_paths[0])
    assert "tal" not in exported.columns
assert not output_dir.exists()
xyz = observations.as_dataset(copy="shallow")[[f"drone.position.{axis}" for axis in "xyz"]].to_array("axis")
expected = np.sqrt((xyz ** 2).sum("axis", skipna=False)).mean("sample")
np.testing.assert_allclose(result["datavar"], expected)
result

/Users/kyle/miniconda3/envs/tal/lib/python3.13/site-packages/zarr/api/asynchronous.py:247: ZarrUserWarning: Consolidated metadata is currently not part in the Zarr format 3 specification. It may not be supported by other zarr implementations and may change in the future.
  warnings.warn(


<tal.AnalysisObject> Size: 192B
Dimensions:  (trial: 12)
TAL:
    Sequence: none
    Batch: (trial)
    Core: ()
Coordinates:
             Role
  * trial    batch  (trial) object 96B 'flight_01' 'flight_02' ... 'flight_12'
Data variables:
    datavar  (trial) float64 96B dask.array<chunksize=(12,), meta=np.ndarray>

<xarray.Dataset> Size: 192B
Dimensions:  (trial: 12)
Coordinates:
  * trial    (trial) object 96B 'flight_01' 'flight_02' ... 'flight_12'
Data variables:
    datavar  (trial) float64 96B 5.922 6.784 6.349 7.26 ... 7.315 7.45 7.472
Attributes:
    tal:      {'version': 1, 'core': {'roles': {'batch_dims': ['trial'], 'cor...

## What each format promises
The decoded Zarr round trip preserves schema and ragged lengths; storage encodings
and chunk layout are implementation choices. CSV is a lossy table export: numerical
columns and timestamps are useful outside TAL, but it does not preserve the complete
schema, component registry, or resources. A later CSV read needs its own declarations.

Keep lazy work inside the resource lifetime. Do not expect a lazy expression to work
after its backing storage is removed. Displays show lazy metadata, whereas plotting
and explicit computation may read payloads.

Condition-driven event discovery has additional restrictions: chunked boundary queries
need bounded capacity where documented, and `around` on a lazy source requires explicit
anchors rather than an unbounded condition scan. This notebook demonstrates selection
and reduction without adding custom discovery logic. Continue to [visualization](13_visualization_holoviews_explorer.ipynb).